In [ ]:
#import 是导入的意思
#requests 是用来访问互联网的：告诉他api地址，他会把请求发给NVD服务器，然后等待服务器返回数据，他在传回python
import requests
#处理数据表格的作用创建表格:处理行和列；读取 CSV；写入 CSV；清洗和分析数据,写作pd是为了更方便。
import pandas as pd

cve_id = "CVE-2026-8931"#(这是一个自设变量名，为的是自己好理解，后面的是漏洞编号)
url = "https://services.nvd.nist.gov/rest/json/cves/2.0"#网址自设变量名url，后面接网址
params = {"cveId": cve_id}#由于cve_id是自设变量名，但由于系统只能识别，所以params类似一个字典，对变量名进行了翻译
#headers是请求头，是用来说明用户代理的
headers = {
    "User-Agent": "nvd-cve-fetcher/1.0"
}

#这里就开始尝试运行了
try:
    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=15
    )
#回复内容要求得到网址（url），参数(params)，请求头(headers)，延迟时间(timeout),左边是固定的参数，右边才是自己前面设置的变量名
    response.raise_for_status()#用于检查服务器返回时的HTTP状态码，成功就会继续运行（200）
    data = response.json()#这是表示把数据都回复成JSON格式，为了方便读取

#这一句表示如果等待时间超时就会停止运行
except requests.exceptions.Timeout:
    print("请求 NVD API 超时")
    raise#raise的功能是如果符合情况就停止运行
#e是一个新变量，把异常都归入篮子里，f的作用是将{}里面的东西在运行时替换成真实值，这里只是为了让代码更加简洁，下面也是这个道理
except requests.exceptions.RequestException as e:
    print(f"请求 NVD API 失败：{e}")
    raise
except ValueError as e:
    print(f"API 返回的内容不是合法 JSON：{e}")
    raise


vulnerabilities = data.get("vulnerabilities", [])#这是取出漏洞列表的意思，前面vulnerabilities是我自设的漏洞名称，可以修改
#后一个vulnerabilities是系统统一的漏洞编码的意思

#不存在漏洞
if not vulnerabilities:
    print(f"没有找到漏洞：{cve_id}")
else:
    cve_item = vulnerabilities[0].get("cve", {})
#存在一个叫cve_item的漏洞是漏洞列表的第一个，表示取出的cve的字段，如果没有就写{}

    # 优先使用 CVSS v4，其次是 v3.1、v3.0、v2（CVSS是漏洞评分系统，第一metrics表示漏洞信息名称，也是我的自设变量名）
    #metrics后一个是系统的自己的漏洞信息总名称，包括score,version. None表示现在还没有分数，也没有具体的哪一个评分系统
    metrics = cve_item.get("metrics", {})
    cvss_score = None
    cvss_version = None

    metric_types = [
        "cvssMetricV40",
        "cvssMetricV31",
        "cvssMetricV30",
        "cvssMetricV2",
    ]

    for metric_type in metric_types:#用上面的四种评分系统都尝试一遍，得到的一个漏洞信息的列表，得出一个评分系统
        metric_list = metrics.get(metric_type, [])

        if metric_list:
            cvss_data = metric_list[0].get("cvssData", {})#评估数据将会在列表中呈现，选择第一个
            cvss_score = cvss_data.get("baseScore")#在自定义的评估数据变量里面找到需要的基础分数和实行的具体的一个评分系统
            cvss_version = cvss_data.get("version")
            break#成功后即可停止

    # 优先选择中文描述，没有中文时使用英文描述
    #这就是将我们找出来的漏洞进行描述，desc是我规定的用于接受最终结果的变量名，next是python的内置函数，用来从生成器里取一个值
    #for in 将descriptions的值赋给了item,lang字段名，意思是语言，zh是中文的字段名，他们在python有固定含义
    descriptions = cve_item.get("descriptions", [])

    desc = next(
        (
            item.get("value")
            for item in descriptions
            if item.get("lang") == "zh"
        ),
        None
    )

    if not desc and descriptions:#如果没有描述或者，没用中文的最终结果，直接输出第一个描述
        desc = descriptions[0].get("value")

    #结果包括了漏洞的id,来源，描述，具体的评估系统，分数，前面这些变量名由于加了双引号，可以直接以原形态输出
    result = {
        "cve_id": cve_item.get("id", cve_id),
        "source": "NVD",
        "description": desc or "",
        "cvss_version": cvss_version or "",
        "cvss_score": cvss_score,
    }

   #这里是将结果保存为CSV文件的意思
   #pd.DataFrame是用来创建表格的，其中DataFrame可以接收多行数据
   #df.to_csv的作用就是把DataFrame保存为CSV文件
    df = pd.DataFrame([result])
    df.to_csv(
        "nvd_result.csv",
        index=False,#因为pandas自带行号，但这不是漏洞的一部分，所以要抹掉
        encoding="utf-8-sig"#使用这个编码保存文件可以梗荣同意正确显示中文
    )

    print("抓取并清洗成功！")
    print(df.to_string(index=False))#显示屏也不会有pandas的序号

